# SageMaker Endpoint 생성 튜토리얼

이 노트북은 Amazon SageMaker를 사용하여 대규모 언어 모델(LLM) 엔드포인트를 생성하는 방법을 단계별로 안내합니다.

## 학습 목표
- SageMaker 환경 설정 및 초기화
- LMI(Large Model Inference) 컨테이너 이해
- vLLM 설정 구성
- 모델 배포 및 엔드포인트 생성
- 리소스 정리

## 사전 요구사항
- AWS 계정 및 SageMaker 접근 권한
- 충분한 GPU 인스턴스 할당량 (ml.g5.xlarge 이상)
- Hugging Face 토큰 (선택사항, 비공개 모델 사용 시)

## 1. 환경 설정 및 초기화

먼저 필요한 라이브러리를 가져오고 SageMaker 환경을 초기화합니다.

In [ ]:
# 필요한 라이브러리 가져오기
import json
from importlib.metadata import version

import boto3

# 🔴 SageMaker Python SDK v3 기준입니다.
#    v2 의 `from sagemaker import Model, image_uris, serializers, deserializers` 와
#    `sagemaker.get_execution_role()` / `sagemaker.session.Session()` 은 v3 에서
#    전부 제거됐습니다(최상위에 남은 것은 core / serve / train / ai_registry 뿐).
#    v3 에서는 아래 경로를 씁니다.
from sagemaker.core.helper.session_helper import Session, get_execution_role
from sagemaker.core.utils import name_from_base

# SageMaker 환경 초기화
# execution role: SageMaker가 다른 AWS 서비스에 접근할 때 사용하는 IAM 역할
role = get_execution_role()

# SageMaker 세션: AWS API와 상호작용하기 위한 세션 객체
sess = Session()

# 기본 S3 버킷: 모델 아티팩트와 기타 파일을 저장할 S3 버킷
bucket = sess.default_bucket()

# 현재 리전과 계정 ID
region = sess.boto_region_name
account_id = sess.account_id()

# AWS 클라이언트 초기화
# SageMaker 클라이언트: 모델, 엔드포인트 등을 관리
sm_client = boto3.client("sagemaker", region_name=region)
# SageMaker Runtime 클라이언트: 엔드포인트 추론 요청
smr_client = boto3.client("sagemaker-runtime", region_name=region)

# 환경 정보 출력
# 🔴 v3 에는 sagemaker.__version__ 이 없습니다. 패키지 메타데이터로 읽습니다.
print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket}")
print(f"sagemaker session region: {region}")
print(f"sagemaker version: {version('sagemaker')}")

## 2. 컨테이너 및 인스턴스 설정

SageMaker에서 대규모 언어 모델을 실행하기 위해 LMI(Large Model Inference) 컨테이너를 사용합니다.

### LMI 컨테이너란?
- AWS에서 제공하는 대규모 모델 추론 전용 컨테이너
- vLLM, TensorRT-LLM 등 최적화된 추론 엔진 포함
- GPU 메모리 효율성과 처리량 최적화

### 인스턴스 타입 선택 가이드
- **ml.g5.xlarge**: 소규모 모델 (7B 이하), 테스트용
- **ml.g6.48xlarge**: 중대형 모델 (13B-70B), 프로덕션 권장
- **ml.p5.48xlarge**: 초대형 모델 (120B+), 최고 성능

In [ ]:
# 리전 설정 - GPU 인스턴스 할당량이 있는 리전 선택
REGION = sess.boto_region_name   # 세션 리전을 그대로 씁니다(불일치 사고 방지)

# LMI 컨테이너 버전 선택
# 최신 버전은 다음 링크에서 확인:
# https://github.com/aws/deep-learning-containers/blob/master/available_images.md#large-model-inference-containers
#
# 🔴 번들 vLLM 버전을 정하는 것은 태그의 lmi<NN> 부분입니다. 앞의 0.36.0 은
#    djl-serving 버전이라 판단 기준이 아닙니다. gemma-4 처럼 최신 아키텍처를 서빙할 때는
#    고정한 태그의 번들 vLLM 버전을 배포 전에 확인하세요(gemma-4 는 vLLM >= 0.19 필요).
#    이 값은 create_endpoint.py 의 DEFAULT_CONTAINER_VERSION 과 같습니다.
CONTAINER_VERSION = "0.36.0-lmi26.0.0-cu130"

# 컨테이너 URI 구성
# 763104351884는 AWS DLC(Deep Learning Container) 계정 ID
container_uri = f'763104351884.dkr.ecr.{REGION}.amazonaws.com/djl-inference:{CONTAINER_VERSION}'

# 인스턴스 타입 선택
# 모델 크기와 예산에 따라 선택
instance_type = "ml.g5.12xlarge"  # gpt-oss-20b 는 TENSOR_PARALLEL_DEGREE=max 로 4장을 씁니다

# 더 큰 모델이나 높은 성능이 필요한 경우:
# instance_type = "ml.g6.48xlarge"  # 대형 모델
# instance_type = "ml.p5.48xlarge"  # 초대형 모델, 최고 성능

print(f"✅ Region: {REGION}")
print(f"📦 Container URI: {container_uri}")
print(f"🖥️ Instance Type: {instance_type}")

# 💡 팁: 처음 사용하는 경우 작은 인스턴스로 시작해서 테스트 후 확장하는 것을 권장합니다.

## 3. 모델 및 vLLM 설정

사용할 모델과 vLLM 추론 엔진의 설정을 구성합니다.

### vLLM이란?
- 대규모 언어 모델을 위한 고성능 추론 엔진
- PagedAttention으로 메모리 효율성 극대화
- 동적 배치 처리로 처리량 최적화

### 주요 설정 옵션 설명
- **TENSOR_PARALLEL_DEGREE**: GPU 간 모델 분산 정도
- **OPTION_ROLLING_BATCH**: 동적 배치 처리 활성화
- **OPTION_ASYNC_MODE**: 비동기 처리 모드

In [ ]:
# 사용할 모델 지정
# Hugging Face Hub에서 공개된 GPT 모델 사용
HF_MODEL_ID = "openai/gpt-oss-20b"

# Hugging Face 토큰 (선택사항)
# 비공개 모델이나 gated 모델 사용 시 필요
# https://huggingface.co/settings/tokens 에서 생성 가능
HF_TOKEN = ""  # 공개 모델이므로 빈 문자열로 설정

# vLLM 설정 구성
vllm_config = {
    # 기본 모델 설정
    "HF_MODEL_ID": HF_MODEL_ID,           # 사용할 모델 ID
    "HF_TOKEN": HF_TOKEN,                 # Hugging Face 인증 토큰
    
    # 타임아웃 설정
    "OPTION_MODEL_LOADING_TIMEOUT": "1500",  # 모델 로딩 타임아웃 (초) - 큰 모델일수록 더 긴 시간 필요
    
    # 오류 처리 설정
    "SERVING_FAIL_FAST": "true",          # 빠른 실패 모드 활성화 - 문제 발생 시 즉시 오류 반환
    
    # 성능 최적화 설정
    "OPTION_ASYNC_MODE": "true",          # 비동기 처리 모드 활성화 -  동시 요청 처리 성능 향상
    "OPTION_ROLLING_BATCH": "disable",    # 롤링 배치 비활성화

    # GPU 병렬 처리 설정
    "TENSOR_PARALLEL_DEGREE": "max",      # 사용 가능한 모든 GPU 활용 - 단일 GPU 인스턴스에서는 "1"과 동일
    
    # 서비스 엔트리포인트
    "OPTION_ENTRYPOINT": "djl_python.lmi_vllm.vllm_async_service"  # vLLM 비동기 서비스 사용
}

print("✅ vLLM 설정 완료")
print(f"📋 모델: {HF_MODEL_ID}")
print(f"🔧 주요 설정:")
print(f"   - 비동기 모드: {vllm_config['OPTION_ASYNC_MODE']}")
print(f"   - 텐서 병렬도: {vllm_config['TENSOR_PARALLEL_DEGREE']}")
print(f"   - 모델 로딩 타임아웃: {vllm_config['OPTION_MODEL_LOADING_TIMEOUT']}초")

## 4. SageMaker 모델 생성

설정한 컨테이너와 vLLM 구성을 사용하여 SageMaker 모델을 생성합니다.

In [ ]:
# 모델 이름 생성
# HF_MODEL_ID에서 모델명만 추출 (예: "openai/gpt-oss-20b" → "gpt-oss-20b")
HF_MODEL_NAME = HF_MODEL_ID.split('/')[-1]

# 🔴 v3 는 `sagemaker.Model` 을 `sagemaker.serve.ModelBuilder` 로 대체했습니다.
#    바뀐 점 세 가지:
#      1) env  -> env_vars,  role -> role_arn
#      2) 컨테이너 이미지와 인스턴스 타입을 생성 시점에 함께 받습니다
#      3) 모델 생성(build)과 배포(deploy)가 분리됐습니다 — build() 를 먼저 부릅니다
from sagemaker.serve import ModelBuilder

model_builder = ModelBuilder(
    image_uri=container_uri,       # 사용할 컨테이너 이미지
    env_vars=vllm_config,          # 환경 변수로 vLLM 설정 전달
    role_arn=role,                 # IAM 실행 역할
    instance_type=instance_type,   # 인스턴스 타입 (v3 는 여기서 받습니다)
)

# SageMaker 모델 리소스를 만듭니다. 엔드포인트는 아직 생기지 않습니다.
model = model_builder.build(model_name=name_from_base(HF_MODEL_NAME))

print("✅ SageMaker 모델 생성 완료")
print(f"🐳 컨테이너: {container_uri}")
print(f"🔑 IAM 역할: {role}")
print("💡 아직 엔드포인트는 없습니다. 다음 셀에서 배포합니다.")

## 5. 엔드포인트 배포

생성한 모델을 실제 추론 엔드포인트로 배포합니다.

### 배포 과정
1. **엔드포인트 구성 생성**: 인스턴스 타입, 개수 등 설정
2. **엔드포인트 생성**: 실제 인프라 프로비저닝
3. **모델 로딩**: 컨테이너에서 모델 다운로드 및 초기화
4. **헬스 체크**: 엔드포인트 정상 동작 확인

⏱️ **예상 소요 시간**: 5-15분 (모델 크기에 따라 다름)

In [ ]:
# 엔드포인트 이름 생성 (타임스탬프 포함으로 고유성 보장)
endpoint_name = name_from_base(HF_MODEL_NAME)

print("🚀 엔드포인트 배포 시작...")
print(f"📍 엔드포인트 이름: {endpoint_name}")
print(f"🖥️ 인스턴스: {instance_type} x 1")
print("⏱️ 예상 소요 시간: 5-15분")
print("")
print("💡 배포 과정:")
print("   1. 엔드포인트 구성 생성")
print("   2. EC2 인스턴스 프로비저닝")
print("   3. 컨테이너 시작 및 모델 로딩")
print("   4. 헬스 체크 완료")
print("")

# 모델 배포 실행
# 🔴 deploy() 는 build() 의 반환값이 아니라 ModelBuilder 에서 호출합니다.
#    build() 가 돌려주는 sagemaker.core.resources.Model 에는 deploy 가 없습니다.
# 🔴 v3 에서 컨테이너 콜드스타트 타임아웃 인자 이름이 바뀌었습니다:
#    container_startup_health_check_timeout -> container_timeout_in_seconds
endpoint = model_builder.deploy(
    initial_instance_count=1,              # 초기 인스턴스 개수
    instance_type=instance_type,           # 인스턴스 타입
    endpoint_name=endpoint_name,           # 엔드포인트 이름
    container_timeout_in_seconds=1800,     # 컨테이너 시작 타임아웃 (30분)
                                           # 큰 모델은 로딩 시간이 오래 걸립니다
    wait=False,                            # 비동기 배포 (즉시 반환)
                                           # True 로 두면 배포 완료까지 대기합니다
)

print("✅ 배포 요청 완료!")
print("📊 배포 상태는 다음 셀에서 확인할 수 있습니다.")

## 6. 배포 상태 모니터링

엔드포인트 배포 진행 상황을 AWS 콘솔에서 확인할 수 있습니다.

In [ ]:
from IPython.display import display, HTML

def make_endpoint_link(region, endpoint_name, endpoint_task):
    """AWS 콘솔 엔드포인트 링크 생성 함수"""
    endpoint_link = f'<b><a target="blank" href="https://console.aws.amazon.com/sagemaker/home?region={region}#/endpoints/{endpoint_name}">{endpoint_task} 엔드포인트 상태 확인</a></b>'   
    return endpoint_link 

# AWS 콘솔 링크 생성 및 표시
endpoint_link = make_endpoint_link(region, endpoint_name, '🔗')
display(HTML(endpoint_link))

print("")
print("📊 배포 상태 확인 방법:")
print("   1. 위 링크를 클릭하여 AWS 콘솔로 이동")
print("   2. 엔드포인트 상태 확인:")
print("      - Creating: 생성 중")
print("      - InService: 서비스 준비 완료 ✅")
print("      - Failed: 배포 실패 ❌")
print("   3. CloudWatch 로그에서 상세 정보 확인 가능")
print("")
print("⚠️ 주의사항:")
print("   - 배포 완료까지 5-15분 소요")
print("   - 모델이 클수록 더 오래 걸림")
print("   - 실패 시 CloudWatch 로그 확인 필요")

## 대안: SageMaker JumpStart 사용 (선택사항)

SageMaker JumpStart를 사용하면 사전 구성된 모델을 더 쉽게 배포할 수 있습니다.

### JumpStart의 장점
- 사전 최적화된 모델 구성
- 원클릭 배포
- Inference Components 지원
- 자동 리소스 할당

아래 코드는 JumpStart를 사용한 배포 예시입니다 (현재는 주석 처리됨).

In [ ]:
# 대안: SageMaker JumpStart 사용 (선택사항)
#
# 🔴 v3 에서 `sagemaker.jumpstart` 모듈은 제거됐습니다. import 하면 다음 오류가 납니다:
#    ModuleNotFoundError: `sagemaker.jumpstart` was removed in the SageMaker Python SDK v3.
#    v3 에서는 ModelBuilder 에 JumpStart 모델 ID 를 넘겨 같은 일을 합니다.
#
# from sagemaker.serve import ModelBuilder
#
# # JumpStart 모델 ID (EULA 동의가 필요한 모델은 약관을 먼저 확인하세요)
# js_model_id = "openai-reasoning-gpt-oss-120b"
#
# js_builder = ModelBuilder(
#     model=js_model_id,                 # v3 는 model 인자로 JumpStart ID 를 받습니다
#     role_arn=role,
#     instance_type="ml.g6.48xlarge",    # 120B 모델에 적합한 인스턴스
# )
# js_builder.build()
# js_endpoint = js_builder.deploy(
#     initial_instance_count=1,
#     instance_type="ml.g6.48xlarge",
#     endpoint_name=name_from_base("gpt-oss-120b"),
#     container_timeout_in_seconds=900,
#     accept_eula=True,                  # 라이선스 조건 확인 후 True
# )
#
# ⚠️ Inference Component 기반 배포(여러 모델을 한 엔드포인트에 호스팅)와 EULA 인자
#    이름은 SDK 버전에 따라 다릅니다. 이 셀은 실행 검증을 하지 않았으므로, 쓰기 전에
#    설치된 버전의 ModelBuilder.deploy 시그니처를 확인하세요:
#      import inspect; from sagemaker.serve import ModelBuilder
#      print(inspect.signature(ModelBuilder.deploy))

## 7. 엔드포인트 삭제

### 🚨 워크샵 완료 후 반드시 실행하세요! 삭제하지 않으면 계속 비용이 발생합니다!🚨

SageMaker 엔드포인트는 삭제하기 전까지 계속 비용이 발생합니다!

### 💰 예상 비용 (ml.g5.xlarge, us-east-1 리전 기준)
- **1시간**: 약 $1.41
- **하루 (24시간)**: $33.84
- **일주일**: $236.88
- **한 달**: $1,015.20

### 🎯 워크샵 완료 체크리스트
- [ ] 엔드포인트 테스트 완료
- [ ] 학습 내용 정리
- [ ] **엔드포인트 삭제 (필수!)**
- [ ] AWS 콘솔에서 삭제 확인

In [ ]:
# 🚨 워크샵 완료 후 반드시 실행하세요! 🚨\n
# 엔드포인트를 삭제하지 않으면 계속 비용이 발생합니다!\n
import time

try:
    response = sm_client.describe_endpoint(EndpointName=endpoint_name)
    current_status = response['EndpointStatus']
    print(f"현재 엔드포인트 상태: {current_status}")

    if current_status in ['InService', 'Failed']:
        print("\n🗑️ 엔드포인트 삭제를 시작합니다...")

        # 엔드포인트 삭제
        print("   1. 엔드포인트 삭제 중...")
        sess.delete_endpoint(endpoint_name)

        # 엔드포인트 구성 삭제
        print("   2. 엔드포인트 구성 삭제 중...")
        sess.delete_endpoint_config(endpoint_name)

        print("\n✅ 리소스 정리 완료!")
        print("💰 더 이상 비용이 발생하지 않습니다. 워크샵을 안전하게 완료했습니다!")

    elif current_status == 'Creating':
        print("⏳ 엔드포인트가 아직 생성 중입니다.")
        print("   생성 완료 후 다시 이 셀을 실행하여 삭제하세요.")

    else:
        print(f"ℹ️ 현재 상태: {current_status}")

except Exception as e:
    if 'Could not find endpoint' in str(e):
        print("✅ 엔드포인트가 이미 삭제되었거나 존재하지 않습니다.")
        print("💰 비용 발생 없음!")
    else:
        print(f"❌ 오류 발생: {e}")
        print("🔧 수동 삭제 방법:")
        print("   1. AWS 콘솔 → SageMaker → 엔드포인트")
        print(f"   2. '{endpoint_name}' 선택 → 삭제")
        print("   3. 엔드포인트 구성도 함께 삭제")